In [1]:
# -*- coding: utf-8 -*-
"""
Pipeline de Análise e Inferência Estatística - Sinistros de Trânsito em Fortaleza (2015-2024)
Pressupõe que a limpeza (ETL) já foi realizada previamente.
"""

import os
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Configurações globais de ambiente e estilo visual
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['font.size'] = 10
os.makedirs('analise_saida', exist_ok=True)


class TrafegoPipeline:

  def __init__(self, filepath='sinistro_transito_final.csv'):
    self.filepath = filepath
    self.df = None

  def etapa_1_ingestao(self):
    print('>>> Etapa 1: Ingestão e Pré-processamento inicial...')
    self.df = pd.read_csv(self.filepath, low_memory=False)
    self.df['data'] = pd.to_datetime(self.df['data'], errors='coerce')
    self.df['ano'] = self.df['data'].dt.year
    print(
        f'Dataset carregado com sucesso. Total de registros: {len(self.df):,}'
    )
    return self

  def etapa_2_analise_descritiva(self):
    print('\n>>> Etapa 2: Estatísticas Descritivas e Temporal...')
    desc = (
        self.df[['mortos', 'feridos', 'ilesos', 'totalveiculos']]
        .describe()
        .T[['count', 'mean', 'std', 'min', 'max']]
    )
    print(desc)
    print('\nDistribuição da Severidade:')
    print(self.df['severidade'].value_counts(dropna=False))
    return self

  def etapa_3_analise_espacial(self):
    print('\n>>> Etapa 3: Análise Espacial (Bairros e Vítimas Fatais)...')
    total_acidentes = len(self.df)
    total_mortos = self.df['mortos'].sum()

    top_acidentes = self.df['bairro'].value_counts().head(5)
    top_mortos = (
        self.df.groupby('bairro')['mortos']
        .sum()
        .sort_values(ascending=False)
        .head(5)
    )

    print('Top 5 Bairros com Mais Acidentes:')
    for bairro, qtd in top_acidentes.items():
      pct = (qtd / total_acidentes) * 100
      print(f'- {bairro}: {qtd} acidentes ({pct:.2f}%)')

    print('\nTop 5 Bairros com Mais Vítimas Fatais:')
    for bairro, mortos in top_mortos.items():
      pct = (mortos / total_mortos) * 100
      print(f'- {bairro}: {mortos} mortos ({pct:.2f}%)')
    return self

  def etapa_4_veiculos_coocorrencia(self):
    print('\n>>> Etapa 4: Frequência e Co-ocorrência de Veículos...')
    veic_cols = [
        'automovel',
        'onibus',
        'bicicleta',
        'caminhao',
        'microonibus',
        'ciclomotor',
        'motocicleta',
        'tracaoanimal',
        'trem',
    ]
    veic_soma = self.df[veic_cols].sum().sort_values(ascending=False)
    print('Soma de veículos envolvidos por tipo:')
    print(veic_soma)

    # Matriz de Co-ocorrência
    binary_df = (self.df[veic_cols] > 0).astype(int)
    cooc_matrix = binary_df.T.dot(binary_df)
    print(
        '\nCo-ocorrência Automóvel x Motocicleta:'
        f" {cooc_matrix.loc['automovel', 'motocicleta']:,}"
    )
    return self

  def etapa_5_testes_inferenciais(self):
    print('\n>>> Etapa 5: Testes Estatísticos (Qui-Quadrado e ANOVA)...')
    # Qui-Quadrado: Severidade x Controle de Tráfego
    ct1 = pd.crosstab(self.df['controletrafego'], self.df['severidade'])
    chi2_1, p_1, _, _ = stats.chi2_contingency(ct1)
    n1 = ct1.sum().sum()
    v_1 = np.sqrt(chi2_1 / (n1 * (min(ct1.shape) - 1)))
    print(
        f"Qui-Quadrado (Controle x Severidade): Chi2={chi2_1:.2f}, p={p_1:.4e},"
        f" V={v_1:.3f}"
    )

    # ANOVA / R²: Feridos x Severidade
    model_sev = ols('feridos ~ C(severidade)', data=self.df).fit()
    print(f'ANOVA R² (Feridos ~ Severidade): {model_sev.rsquared:.4f}')
    return self

  def etapa_6_verificacao_hipoteses(self):
    print('\n>>> Etapa 6: Validação das 5 Hipóteses Científicas...')

    # H1: Fatais na madrugada / fim de semana
    self.df['fatal'] = (self.df['severidade'] == 'fatal').astype(int)
    self.df['madr_fds'] = (
        self.df['faixahoraria'].str.contains('Madrugada', na=False)
        | (self.df['fimdesemana'] == 'sim')
    ).astype(int)
    ct_h1 = pd.crosstab(self.df['madr_fds'], self.df['fatal'])
    chi2_h1, p_h1, _, _ = stats.chi2_contingency(ct_h1)
    print(f'H1 (Fatais em Madrugada/FDS) - p-valor: {p_h1:.4e}')

    # H3: Tendência Linear Mensal (R²)
    mensal = (
        self.df.groupby(self.df['data'].dt.to_period('M'))
        .size()
        .reset_index(name='acidentes')
    )
    mensal['tempo'] = np.arange(len(mensal))
    X_h3 = sm.add_constant(mensal['tempo'])
    model_h3 = sm.OLS(mensal['acidentes'], X_h3).fit()
    print(f'H3 (Tendência Temporal Mensal) - R²: {model_h3.rsquared:.3f}')

    # H5: Regressão Múltipla por Bairro
    bairro_df = (
        self.df.groupby('bairro')
        .agg(
            acidentes=('data', 'count'),
            motos=('motocicleta', 'sum'),
            prop_cruz=('tipocruzamento', lambda x: (x != 'meio de quadra').mean()),
        )
        .reset_index()
    )
    X_h5 = sm.add_constant(bairro_df[['motos', 'prop_cruz']])
    model_h5 = sm.OLS(bairro_df['acidentes'], X_h5).fit()
    print(
        f'H5 (Modelo Espacial por Bairros) - R²: {model_h5.rsquared:.3f} |'
        f' F-pvalue: {model_h5.f_pvalue:.4e}'
    )
    print('\nPipeline executado com sucesso e ponta a ponta!')
    return self


# Execução do Pipeline
if __name__ == '__main__':
  pipeline = TrafegoPipeline()
  (
      pipeline.etapa_1_ingestao()
      .etapa_2_analise_descritiva()
      .etapa_3_analise_espacial()
      .etapa_4_veiculos_coocorrencia()
      .etapa_5_testes_inferenciais()
      .etapa_6_verificacao_hipoteses()
  )

>>> Etapa 1: Ingestão e Pré-processamento inicial...
Dataset carregado com sucesso. Total de registros: 130,820

>>> Etapa 2: Estatísticas Descritivas e Temporal...
                  count      mean       std  min   max
mortos         130820.0  0.015181  0.126090  0.0   3.0
feridos        130820.0  1.069737  0.702907  0.0  26.0
ilesos         130820.0  0.527725  0.819738  0.0   8.0
totalveiculos  130820.0  1.619034  0.729245  0.0  19.0

Distribuição da Severidade:
severidade
ferido           94597
ileso            25055
não informado     9242
fatal             1926
Name: count, dtype: int64

>>> Etapa 3: Análise Espacial (Bairros e Vítimas Fatais)...
Top 5 Bairros com Mais Acidentes:
- centro: 7464 acidentes (5.71%)
- aldeota: 5284 acidentes (4.04%)
- meireles: 4098 acidentes (3.13%)
- parangaba: 3524 acidentes (2.69%)
- fátima: 3378 acidentes (2.58%)

Top 5 Bairros com Mais Vítimas Fatais:
- centro: 73 mortos (3.68%)
- cajazeiras: 61 mortos (3.07%)
- messejana: 53 mortos (2.67%)
- pre